# 03 — RAG Evaluation

Evaluate the full pipeline on a set of gold Q&A pairs. We compare:
- **Context Relevance** — how well retrieved chunks match the question
- **Answer Faithfulness** — how grounded the answer is in the context
- **Answer Relevancy** — how well the answer addresses the question

We also sweep over `k` (number of retrieved chunks) to observe the trade-off.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.pipeline import RAGPipeline
from src.evaluation.evaluator import RAGEvaluator

## 1. Gold Q&A Pairs

In [ ]:
QA_PAIRS = [
    {
        'question': 'What is machine learning?',
        'reference': 'Machine learning is a subset of AI that allows systems to learn from data and improve without explicit programming.',
    },
    {
        'question': 'What are the three types of machine learning?',
        'reference': 'Supervised learning, unsupervised learning, and reinforcement learning.',
    },
    {
        'question': 'What is a neural network?',
        'reference': 'An artificial neural network is a computational model inspired by the brain, consisting of layers of interconnected nodes.',
    },
    {
        'question': 'What is Retrieval-Augmented Generation?',
        'reference': 'RAG combines information retrieval with generative language models to improve factual accuracy.',
    },
    {
        'question': 'What are the applications of AI in healthcare?',
        'reference': 'AI assists radiologists in detecting tumors, predicts patient deterioration, and accelerates drug discovery.',
    },
]
print(f'Evaluating on {len(QA_PAIRS)} Q&A pairs.')

## 2. Build Pipeline

In [ ]:
pipeline = RAGPipeline(docs_dir='../data/sample_docs', mode='local', k=5)
print(f'Indexed vectors: {len(pipeline.vector_store)}')

## 3. Run Evaluation

In [ ]:
results = []

for qa in QA_PAIRS:
    question = qa['question']
    result = pipeline.query(question)
    scores = result['scores']
    results.append({
        'Question': question,
        'Answer (truncated)': result['answer'][:80] + '…',
        'Context Relevance': scores['context_relevance'],
        'Answer Faithfulness': scores['answer_faithfulness'],
        'Answer Relevancy': scores['answer_relevancy'],
        'Overall': scores['overall'],
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

## 4. Metrics Summary Table

In [ ]:
summary = df[['Context Relevance', 'Answer Faithfulness', 'Answer Relevancy', 'Overall']].describe()
print('\n=== Metric Summary ===' )
print(summary.loc[['mean', 'std', 'min', 'max']].round(4).to_string())

## 5. Per-Question Metric Bar Chart

In [ ]:
metric_cols = ['Context Relevance', 'Answer Faithfulness', 'Answer Relevancy']
short_questions = [q[:35] + '…' for q in df['Question']]

x = np.arange(len(df))
width = 0.25
colors = ['steelblue', 'seagreen', 'tomato']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (col, color) in enumerate(zip(metric_cols, colors)):
    ax.bar(x + i * width, df[col], width, label=col, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(short_questions, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_title('RAG Evaluation Metrics per Question', fontweight='bold')
ax.legend()
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.show()

## 6. Effect of k on Retrieval Quality

In [ ]:
k_values = [1, 2, 3, 5]
k_results = {k: [] for k in k_values}

evaluator = RAGEvaluator(embedder=pipeline.embedder)

for k in k_values:
    pipeline.k = k
    for qa in QA_PAIRS:
        result = pipeline.query(qa['question'])
        k_results[k].append(result['scores']['context_relevance'])

mean_cr = {k: np.mean(v) for k, v in k_results.items()}
std_cr  = {k: np.std(v)  for k, v in k_results.items()}

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    list(mean_cr.keys()),
    list(mean_cr.values()),
    yerr=list(std_cr.values()),
    marker='o', color='steelblue', linewidth=2, capsize=5, capthick=2,
)
ax.set_xlabel('k (number of retrieved chunks)')
ax.set_ylabel('Mean Context Relevance')
ax.set_title('Context Relevance vs. k', fontweight='bold')
ax.set_ylim(0, 1)
ax.set_xticks(k_values)
plt.tight_layout()
plt.show()

print('k  | mean_context_relevance')
print('---|------------------------')
for k in k_values:
    print(f'{k:2d} | {mean_cr[k]:.4f} ± {std_cr[k]:.4f}')